# Group 4 — Advanced Custom Example (v2)
## Memetic GA + 2-opt: Philippine Aviation Route Optimization (TSP)
### Real Aviation Waypoints | Cartopy Maps (with fallback) | 2-opt Local Search

**Textbook:** Kuznetsov, O. (2026). *Intelligent Systems: From Theory to Applications.* Springer.

---

### Why v2: Memetic Algorithm (GA + 2-opt)
In a plain GA, HC beats GA on small TSP because HC does an exhaustive neighborhood scan per iteration. The fix is a **Memetic Algorithm** — apply 2-opt local search to every GA offspring.

> *"Memetic Algorithms combine genetic search with local optimization."* — Kuznetsov, Ch. 9, Sec. 9.4.4, p. 201

**Cartopy note:** Maps use Cartopy real coastlines in Colab (needs network on first run to download Natural Earth shapefiles). Offline fallback draws island outlines from convex hulls of the waypoint dataset itself.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS, DATASET, CARTOPY SETUP
# ============================================================
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.spatial import ConvexHull
import random, math, warnings
warnings.filterwarnings('ignore')

# ── Cartopy detection ─────────────────────────────────────────
HAS_CARTOPY = False
try:
    import cartopy.crs as ccrs # import here to catch ImportError if cartopy isn't installed
    import cartopy.feature as cfeature # for land features
    matplotlib.use('Agg') # use non-interactive backend to avoid display issues in Colab without internet
    _f, _a = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}) # subploits
    _a.set_extent([116,118,14,16])
    _a.add_feature(cfeature.LAND)
    _f.canvas.draw()
    plt.close(_f)
    HAS_CARTOPY = True
    print('Cartopy OK — real coastlines enabled')
except Exception as e:
    print(f'Cartopy offline fallback active ({type(e).__name__})')
    print('Run in Colab with internet for real coastlines.')

# ── Dataset ───────────────────────────────────────────────────
df = pd.read_csv('global_aviation_waypoints.csv')
df = df[
    (df['LATITUDE']  >= -90) & (df['LATITUDE']  <= 90) &
    (df['LONGITUDE'] >= -180) & (df['LONGITUDE'] <= 180)
].reset_index(drop=True)

ph_df = df[df['COUNTRY_CODE'] == 'PH'].reset_index(drop=True)
random.seed(42); np.random.seed(42)
WAYPOINTS = ph_df.sample(55, random_state=42).reset_index(drop=True)
CITIES    = WAYPOINTS[['IDENT','LATITUDE','LONGITUDE']].values
N         = len(CITIES)

# ── Island outlines from convex hulls (fallback landmasses) ──
ph_pts  = ph_df[['LATITUDE','LONGITUDE']].values
ISLANDS = {}
for name, lo, hi in [('Luzon',13,22), ('Visayas',9,13), ('Mindanao',4,9)]:
    pts = ph_pts[(ph_pts[:,0]>lo) & (ph_pts[:,0]<=hi)]
    if len(pts) > 3:
        hull  = ConvexHull(pts[:,[1,0]])
        ISLANDS[name] = pts[:,[1,0]][hull.vertices]

print(f'\nWaypoints loaded : {len(df):,} across {df["COUNTRY_NAME"].nunique()} countries')
print(f'PH waypoints     : {len(ph_df)}')
print(f'TSP cities (N)   : {N}')
print(f'Possible routes  : {math.factorial(N):,.0f}')
print()
print(WAYPOINTS[['IDENT','LATITUDE','LONGITUDE']].to_string(index=False))

Cartopy OK — real coastlines enabled

Waypoints loaded : 50,792 across 209 countries
PH waypoints     : 373
TSP cities (N)   : 55
Possible routes  : 12,696,403,353,658,276,446,882,823,840,816,011,312,245,221,598,828,319,560,272,916,152,712,167,424

IDENT  LATITUDE  LONGITUDE
TABUL  6.543503 121.882297
BAYAN 14.831336 120.655697
ASTUR 10.617564 123.697572
SBA65 14.851647 120.391931
CGO19  8.350422 124.594022
NOYAN 10.405822 121.004294
CIA80 15.211369 120.572239
GARAY 14.912322 121.148872
SBA23 14.632764 120.008811
GSA17  6.037708 125.098706
MUPOB 15.636614 123.054969
BOHOL  9.571681 124.379603
KARAG 14.433028 119.945536
CGO15  8.577431 124.652694
KAWIT 10.771883 124.504847
R2683  7.116675 125.635489
ABLBG 13.843603 120.113717
MUMOT 19.028583 117.789725
TALIM 14.356586 121.328114
DILIS 14.518636 126.001419
ANKUR  6.658083 121.001386
MABID  9.995158 124.092969
CIA16 15.149942 120.693144
NIBAT  7.464778 125.623447
PARAL  7.289242 122.261569
GULFO 10.382100 122.706422
MINDA 14.468886 120.45

In [3]:
# ============================================================
# CELL 2 — MAP HELPER (Cartopy or plain matplotlib fallback)
# ============================================================

def make_ph_ax(fig, spec, extent=[115, 129, 3, 22]):
    """Returns axes with PH landmasses — Cartopy if available, hull fallback otherwise."""
    if HAS_CARTOPY:
        ax = fig.add_subplot(spec, projection=ccrs.PlateCarree())
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN,     facecolor='#c8dff0', zorder=0)
        ax.add_feature(cfeature.LAND,      facecolor='#e8e0d0', zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor='#555', zorder=2)
        ax.add_feature(cfeature.LAKES,     facecolor='#c8dff0', zorder=2)
        ax.add_feature(cfeature.RIVERS,    linewidth=0.3, edgecolor='#88aacc', zorder=2)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                          alpha=0.5, linestyle='--')
        gl.top_labels = False; gl.right_labels = False
        ax._use_cartopy = True
    else:
        ax = fig.add_subplot(spec)
        ax.set_facecolor('#c8dff0')
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        for verts in ISLANDS.values():
            poly = plt.Polygon(verts, facecolor='#e8e0d0',
                               edgecolor='#666', linewidth=0.8, alpha=0.85, zorder=1)
            ax.add_patch(poly)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.set_xlabel('Longitude', fontsize=8)
        ax.set_ylabel('Latitude',  fontsize=8)
        ax._use_cartopy = False
    return ax

def gplot(ax, lons, lats, **kw):
    if getattr(ax, '_use_cartopy', False):
        return ax.plot(lons, lats, transform=ccrs.PlateCarree(), **kw)
    return ax.plot(lons, lats, **kw)

def gscatter(ax, lons, lats, **kw):
    if getattr(ax, '_use_cartopy', False):
        return ax.scatter(lons, lats, transform=ccrs.PlateCarree(), **kw)
    return ax.scatter(lons, lats, **kw)

def gannotate(ax, txt, lon, lat, **kw):
    if getattr(ax, '_use_cartopy', False):
        return ax.annotate(txt, (lon, lat), transform=ccrs.PlateCarree(), **kw)
    return ax.annotate(txt, (lon, lat), **kw)

print('Map helper ready.')
print('Mode:', 'Cartopy real coastlines' if HAS_CARTOPY else 'Matplotlib hull fallback')

Map helper ready.
Mode: Cartopy real coastlines


In [4]:
# ============================================================
# CELL 3 — FIGURE 1: GLOBAL MAP + PH ZOOM
# ============================================================
fig = plt.figure(figsize=(18, 8))
gs  = GridSpec(1, 2, figure=fig, width_ratios=[2.2, 1])

# Left: world scatter (always plain matplotlib)
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor('#c8dff0')
ax1.scatter(df['LONGITUDE'], df['LATITUDE'],
            s=0.3, alpha=0.18, color='steelblue', label=f'All waypoints ({len(df):,})')
ax1.scatter(ph_df['LONGITUDE'], ph_df['LATITUDE'],
            s=8, alpha=0.9, color='darkorange', zorder=5, label=f'Philippines ({len(ph_df)})')
ax1.scatter(WAYPOINTS['LONGITUDE'], WAYPOINTS['LATITUDE'],
            s=80, color='crimson', marker='*', zorder=6, label=f'Our {N} TSP waypoints')
ax1.set_xlim(-180,180); ax1.set_ylim(-90,90)
ax1.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax1.axvline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax1.set_xlabel('Longitude', fontsize=11); ax1.set_ylabel('Latitude', fontsize=11)
ax1.set_title('Global Aviation Waypoints Dataset\n(51,043 real ICAO/FAA waypoints, 209 countries)',
              fontsize=12, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9); ax1.grid(True, alpha=0.2)

# Right: PH zoom with landmasses
ax2 = make_ph_ax(fig, gs[1])
gscatter(ax2, ph_df['LONGITUDE'].values, ph_df['LATITUDE'].values,
         s=12, alpha=0.4, color='darkorange', zorder=3)
gscatter(ax2, WAYPOINTS['LONGITUDE'].values, WAYPOINTS['LATITUDE'].values,
         s=130, color='crimson', marker='*', zorder=4)
for _, r in WAYPOINTS.iterrows():
    gannotate(ax2, r['IDENT'], r['LONGITUDE'], r['LATITUDE'],
              fontsize=6.5, color='darkred', ha='left',
              xytext=(3,3), textcoords='offset points')
ax2.set_title(f'Philippine Waypoints — TSP Problem Space\n(373 PH waypoints + {N} selected)',
              fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('fig1_dataset_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

Figure 1 saved.


In [5]:
# ============================================================
# CELL 4 — FIGURE 2: DATASET STATISTICS (3-panel)
# ============================================================
top15   = df['COUNTRY_NAME'].value_counts().head(15)
all_cnt = df['COUNTRY_NAME'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = plt.cm.Blues(np.linspace(0.35, 0.9, 15))[::-1]
bars = axes[0].barh(top15.index[::-1], top15.values[::-1], color=colors, edgecolor='white')
for bar, v in zip(bars, top15.values[::-1]):
    axes[0].text(bar.get_width()+40, bar.get_y()+bar.get_height()/2, f'{v:,}', va='center', fontsize=8.5)
axes[0].set_xlim(0, 6200); axes[0].set_xlabel('Waypoint count', fontsize=10)
axes[0].set_title('Top 15 Countries by Waypoint Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

axes[1].hist(all_cnt.values, bins=40, color='steelblue', edgecolor='white', alpha=0.8, log=True)
axes[1].axvline(len(ph_df), color='crimson', lw=2, ls='--', label=f'Philippines ({len(ph_df)})')
axes[1].set_xlabel('Waypoints per country', fontsize=10)
axes[1].set_ylabel('Countries (log scale)', fontsize=10)
axes[1].set_title('Distribution Across 209 Countries', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

axes[2].hist(df['LATITUDE'], bins=60, color='teal', edgecolor='white', alpha=0.8, orientation='horizontal')
axes[2].axhline(ph_df['LATITUDE'].mean(), color='crimson', lw=2, ls='--',
                label=f'PH avg lat ({ph_df["LATITUDE"].mean():.1f}°)')
axes[2].set_ylabel('Latitude (°)', fontsize=10); axes[2].set_xlabel('Waypoint count', fontsize=10)
axes[2].set_title('Waypoint Density by Latitude\n(northern hemisphere bias)', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig2_dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

Figure 2 saved.


In [6]:
# ============================================================
# CELL 5 — DISTANCE MATRIX + HEATMAP
# ============================================================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin((phi2-phi1)/2)**2
         + math.cos(phi1)*math.cos(phi2)*math.sin(math.radians(lon2-lon1)/2)**2)
    return 2*R*math.asin(math.sqrt(a))

DIST = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            DIST[i][j] = haversine(float(CITIES[i][1]), float(CITIES[i][2]),
                                   float(CITIES[j][1]), float(CITIES[j][2]))
labels = [c[0] for c in CITIES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im = axes[0].imshow(DIST, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Distance (km)')
axes[0].set_xticks(range(N)); axes[0].set_yticks(range(N))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=7.5)
axes[0].set_yticklabels(labels, fontsize=7.5)
axes[0].set_title('Pairwise Haversine Distance Matrix (km)', fontsize=11, fontweight='bold')
for i in range(N):
    for j in range(N):
        if i != j and DIST[i][j] < 200:
            axes[0].text(j, i, f'{DIST[i][j]:.0f}', ha='center', va='center',
                         fontsize=5, color='navy', fontweight='bold')

nn_dists = [min(DIST[i][j] for j in range(N) if j!=i) for i in range(N)]
col_nn   = plt.cm.RdYlGn_r(np.array(nn_dists)/max(nn_dists))
bars = axes[1].barh(labels, nn_dists, color=col_nn, edgecolor='white')
for bar, v in zip(bars, nn_dists):
    axes[1].text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
                 f'{v:.0f} km', va='center', fontsize=8)
axes[1].set_xlabel('Distance to nearest neighbor (km)', fontsize=10)
axes[1].set_title('Nearest-Neighbor Distance per Waypoint\n(red=isolated, green=clustered)',
                  fontsize=11, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_distance_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')
print(f'Most isolated: {labels[nn_dists.index(max(nn_dists))]} ({max(nn_dists):.0f} km to nearest)')

Figure 3 saved.
Most isolated: DILIS (340 km to nearest)


In [7]:
# ============================================================
# CELL 6 — GA + 2-OPT COMPONENTS
# (Memetic Algorithm: Ch. 9, Sec. 9.4.4, p. 201)
# ============================================================

def route_distance(route):
    return (sum(DIST[route[i]][route[i+1]] for i in range(N-1))
            + DIST[route[-1]][route[0]])

def fitness(route): return -route_distance(route)

def init_pop(ps): return [random.sample(range(N), N) for _ in range(ps)]

def tournament_select(pop, k=5):
    return max(random.sample(pop, k), key=fitness)

def ordered_crossover(p1, p2):
    """OX — no duplicate cities. Ch. 9, Sec. 9.2.4, p. 193"""
    a, b = sorted(random.sample(range(N), 2))
    child = [-1]*N
    child[a:b] = p1[a:b]
    rem = [x for x in p2 if x not in child]
    ptr = 0
    for i in range(N):
        if child[i] == -1: child[i] = rem[ptr]; ptr += 1
    return child

def swap_mutate(route, rate=0.02):
    """Swap mutation. Ch. 9, Sec. 9.2.5, p. 194"""
    r = route[:]
    for i in range(N):
        if random.random() < rate:
            j = random.randint(0, N-1)
            r[i], r[j] = r[j], r[i]
    return r

def two_opt(route):
    """
    2-opt local search: try reversing every sub-segment;
    keep any improvement; repeat until no improvement.
    LOCAL SEARCH half of the Memetic Algorithm.
    (Ch. 9, Sec. 9.4.4, p. 201)
    """
    best = route[:]
    improved = True
    while improved:
        improved = False
        for i in range(1, N-1):
            for j in range(i+1, N):
                new = best[:i] + best[i:j+1][::-1] + best[j+1:]
                if route_distance(new) < route_distance(best):
                    best = new; improved = True
    return best

print('Memetic GA operators ready:')
print('  Permutation encoding  — Ch. 9, Sec. 9.2.1, p. 192')
print('  Negative-dist fitness — Ch. 9, Sec. 9.5.1, p. 203')
print('  Tournament select k=5 — Ch. 9, Sec. 9.2.3, p. 193')
print('  Ordered Crossover OX  — Ch. 9, Sec. 9.2.4, p. 193')
print('  Swap mutation r=0.02  — Ch. 9, Sec. 9.2.5, p. 194')
print('  2-opt local search    — Ch. 9, Sec. 9.4.4, p. 201  ← Memetic')

Memetic GA operators ready:
  Permutation encoding  — Ch. 9, Sec. 9.2.1, p. 192
  Negative-dist fitness — Ch. 9, Sec. 9.5.1, p. 203
  Tournament select k=5 — Ch. 9, Sec. 9.2.3, p. 193
  Ordered Crossover OX  — Ch. 9, Sec. 9.2.4, p. 193
  Swap mutation r=0.02  — Ch. 9, Sec. 9.2.5, p. 194
  2-opt local search    — Ch. 9, Sec. 9.4.4, p. 201  ← Memetic


In [ ]:
# ============================================================
# CELL 7 — HC BASELINE
# (Ch. 10, Sec. 10.4, pp. 221-224)
# ============================================================
random.seed(42)

def hc_tsp(n_restarts=10):
    best_route, best_dist = None, float('inf')
    history = []

    # designed to be in a loop for multiple restarts, but single 
    # restart is enough to show local optima
    for _ in range(n_restarts):
        cur = random.sample(range(N), N) # 1. initial state line for hc
        # b. save distance as [a] (curent distance)
        cur_d = route_distance(cur)
        improved = True

        #f. repeat until no improvement.
        while improved:
            improved = False 
            
            # #2. evaluate all neighboring states
            for i in range(N-1): # 55-1 until 0
                for j in range(i+1, N):
                    nb = cur[:];  # creating a full copy ng current route
                    nb[i],nb[j] = nb[j],nb[i]
                    # a. compute distance between initial and neighboring state
                    nb_d = route_distance(nb)
                    # c. compare [a] to next [b] and so on
                    if nb_d < cur_d: 
                        cur, cur_d = nb, nb_d; # d. save shortest distance [c] to [a]
                        improved = True # e. current distance becomes [a]
        history.append(cur_d)

        #3. select best neighbor.
        if cur_d < best_dist: 
            best_dist, best_route = cur_d, cur[:]
    return best_route, best_dist, history

print('Running HC (10 restarts)...')
hc_route, hc_dist, hc_hist = hc_tsp(10)
print(f'HC best  : {hc_dist:.1f} km')
print(f'HC worst : {max(hc_hist):.1f} km  |  std: {np.std(hc_hist):.1f} km')

fig = plt.figure(figsize=(15, 5))
gs2 = GridSpec(1, 2, figure=fig)

ax_bar = fig.add_subplot(gs2[0])
bar_colors = ['crimson' if d==min(hc_hist) else 'steelblue' for d in hc_hist]
ax_bar.bar(range(1,11), hc_hist, color=bar_colors, edgecolor='white')
ax_bar.axhline(min(hc_hist),     color='green',  ls='--', lw=1.8, label=f'Best  {min(hc_hist):.0f} km')
ax_bar.axhline(np.mean(hc_hist), color='orange', ls='--', lw=1.8, label=f'Mean  {np.mean(hc_hist):.0f} km')
for i,v in enumerate(hc_hist): ax_bar.text(i+1, v+20, f'{v:.0f}', ha='center', fontsize=8)
ax_bar.set_xlabel('Restart #', fontsize=11); ax_bar.set_ylabel('Route distance (km)', fontsize=11)
ax_bar.set_title('HC Random-Restart: Each Lands on a Different Local Optimum\n(Ch. 10, Sec. 10.4, pp. 221-224)',
                 fontsize=11, fontweight='bold')
ax_bar.legend(fontsize=10); ax_bar.grid(axis='y', alpha=0.3); ax_bar.set_xticks(range(1,11))

ax_hmap = make_ph_ax(fig, gs2[1])
r_lons = [float(CITIES[i][2]) for i in hc_route+[hc_route[0]]]
r_lats = [float(CITIES[i][1]) for i in hc_route+[hc_route[0]]]
gplot(ax_hmap, r_lons, r_lats, color='steelblue', lw=1.5, marker='o', ms=5, alpha=0.8, zorder=3)
gscatter(ax_hmap, [r_lons[0]], [r_lats[0]], color='green', s=150, zorder=5, marker='*')
for i, idx in enumerate(hc_route):
    gannotate(ax_hmap, f'{i+1}.{CITIES[idx][0]}',
              float(CITIES[idx][2]), float(CITIES[idx][1]),
              fontsize=6, xytext=(4,2), textcoords='offset points')
ax_hmap.set_title(f'HC Best Route: {hc_dist:.0f} km (local optimum)\nMemetic GA will beat this',
                  fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('fig4_hc_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

Running HC (10 restarts)...
HC best  : 8134.4 km
HC worst : 13177.2 km  |  std: 1247.9 km
Figure 4 saved.


In [ ]:
# ============================================================
# CELL 8 — MEMETIC GA EVOLUTION LOOP
# (Ch. 9, Sec. 9.4.4, p. 201)
# ============================================================
POP_SIZE  = 20
GENS      = 30
CR        = 0.85
MR        = 0.02
ELITES    = 3
TOURNEY_K = 5
SNAP_GENS = {1, 10, 30, 60, 100, 150, 200}

random.seed(42); np.random.seed(42)
print('Warm-starting: 2-opt on initial population...')
population = [two_opt(r) for r in init_pop(POP_SIZE)] # genetic algorithm initial state (memetic one)
print('Done.\n')
print(f'{"Gen":>5} | {"Best km":>9} | {"Avg km":>9} | {"Diversity":>10}')
print('-'*42)

best_hist, avg_hist, worst_hist, div_hist, snapshots = [], [], [], [], {}

for gen in range(1, GENS+1):
    population.sort(key=fitness, reverse=True)
    dists = [route_distance(r) for r in population]
    best_hist.append(min(dists)); avg_hist.append(np.mean(dists))
    worst_hist.append(max(dists))

    s   = population[:20]
    div = np.mean([sum(a!=b for a,b in zip(s[i],s[j]))
                   for i in range(len(s)) for j in range(i+1,len(s))])
    div_hist.append(div)

    if gen in SNAP_GENS: snapshots[gen] = population[0][:]

    new_pop = population[:ELITES]
    while len(new_pop) < POP_SIZE:
        p1 = tournament_select(population, TOURNEY_K)
        p2 = tournament_select(population, TOURNEY_K)
        child = ordered_crossover(p1, p2) if random.random() < CR else p1[:]
        child = two_opt(swap_mutate(child, MR))   # ← 2-opt on every offspring
        new_pop.append(child)
    population = new_pop

    if gen % 40 == 0 or gen == 1:
        print(f'{gen:>5} | {min(dists):>9.1f} | {np.mean(dists):>9.1f} | {div:>10.2f}')

population.sort(key=fitness, reverse=True)
ga_route = population[0]
ga_dist  = route_distance(ga_route)

print(f'\nMemetic GA best : {ga_dist:.1f} km')
print(f'HC best         : {hc_dist:.1f} km')
diff = hc_dist - ga_dist
if diff > 0: print(f'GA wins by      : {diff:.1f} km ({diff/hc_dist*100:.1f}% shorter)')
else:        print(f'Tied / marginal — try more GENS')

Warm-starting: 2-opt on initial population...
Done.

  Gen |   Best km |    Avg km |  Diversity
------------------------------------------
    1 |    6133.8 |    6461.4 |      54.04

Memetic GA best : 6011.8 km
HC best         : 8134.4 km
GA wins by      : 2122.6 km (26.1% shorter)


In [11]:
# ============================================================
# CELL 9 — FIGURE 5: CONVERGENCE (4-panel)
# ============================================================
gens_range = range(1, GENS+1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(gens_range, best_hist,  color='green',     lw=2,   label='Best')
axes[0,0].plot(gens_range, avg_hist,   color='steelblue', lw=1.5, ls='--', label='Average')
axes[0,0].plot(gens_range, worst_hist, color='salmon',    lw=1,   ls=':',  alpha=0.8, label='Worst')
axes[0,0].axhline(hc_dist, color='darkorange', lw=1.8, ls='-.', label=f'HC best ({hc_dist:.0f} km)')
axes[0,0].axhline(ga_dist,  color='darkgreen',  lw=1.5, ls=':',  label=f'GA final ({ga_dist:.0f} km)')
axes[0,0].fill_between(gens_range, best_hist, avg_hist, alpha=0.1, color='green')
axes[0,0].set_xlabel('Generation', fontsize=11); axes[0,0].set_ylabel('Route Distance (km)', fontsize=11)
axes[0,0].set_title('Memetic GA Convergence\n(2-opt applied to every offspring)',
                    fontsize=11, fontweight='bold')
axes[0,0].legend(fontsize=9); axes[0,0].grid(True, alpha=0.3)

deltas = [best_hist[i-1]-best_hist[i] for i in range(1,len(best_hist))]
axes[0,1].bar(range(1,len(deltas)+1), deltas,
              color=['green' if d>0 else 'lightgray' for d in deltas],
              alpha=0.8, width=1.0)
axes[0,1].set_xlabel('Generation', fontsize=11)
axes[0,1].set_ylabel('Improvement (km)', fontsize=11)
axes[0,1].set_title('Per-Generation Improvement', fontsize=11, fontweight='bold')
axes[0,1].grid(axis='y', alpha=0.3)

axes[1,0].plot(gens_range, div_hist, color='purple', lw=1.8)
axes[1,0].fill_between(gens_range, div_hist, alpha=0.15, color='purple')
axes[1,0].set_xlabel('Generation', fontsize=11)
axes[1,0].set_ylabel('Avg pairwise position diff', fontsize=10)
axes[1,0].set_title('Population Diversity Over Time', fontsize=11, fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

random.seed(42); np.random.seed(42)
pop_c = [two_opt(r) for r in init_pop(POP_SIZE)]
pop_snaps = {}
for g in range(1, GENS+1):
    pop_c.sort(key=fitness, reverse=True)
    if g in {1, 50, 100, 200}: pop_snaps[g] = [route_distance(r) for r in pop_c]
    np2 = pop_c[:ELITES]
    while len(np2) < POP_SIZE:
        p1=tournament_select(pop_c,TOURNEY_K); p2=tournament_select(pop_c,TOURNEY_K)
        ch = ordered_crossover(p1,p2) if random.random()<CR else p1[:]
        np2.append(two_opt(swap_mutate(ch,MR)))
    pop_c = np2

for g, c in zip(sorted(pop_snaps.keys()), ['#d73027','#fc8d59','#91bfdb','#4575b4']):
    axes[1,1].hist(pop_snaps[g], bins=15, alpha=0.55, color=c, edgecolor='white', label=f'Gen {g}')
axes[1,1].axvline(hc_dist, color='black', lw=1.5, ls='--', label='HC best')
axes[1,1].set_xlabel('Route Distance (km)', fontsize=11)
axes[1,1].set_ylabel('Count', fontsize=11)
axes[1,1].set_title('Population Fitness Distribution at Key Generations\n(left shift = improving)',
                    fontsize=11, fontweight='bold')
axes[1,1].legend(fontsize=9); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig5_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

Figure 5 saved.


In [12]:
# ============================================================
# CELL 10 — FIGURE 6: ROUTE EVOLUTION SNAPSHOTS (PH map)
# ============================================================
snap_keys = sorted(snapshots.keys())
ncols = 4
nrows = math.ceil(len(snap_keys)/ncols)
fig   = plt.figure(figsize=(20, 5*nrows))
gs_s  = GridSpec(nrows, ncols, figure=fig, hspace=0.4, wspace=0.3)
cmap_legs = plt.cm.RdYlGn

for idx, gen in enumerate(snap_keys):
    row, col  = divmod(idx, ncols)
    route, dist = snapshots[gen], route_distance(snapshots[gen])
    ax = make_ph_ax(fig, gs_s[row, col])

    for k in range(N):
        x0=float(CITIES[route[k]][2]);       y0=float(CITIES[route[k]][1])
        x1=float(CITIES[route[(k+1)%N]][2]); y1=float(CITIES[route[(k+1)%N]][1])
        color = cmap_legs(1 - DIST[route[k]][route[(k+1)%N]]/max(DIST.flatten()))
        gplot(ax, [x0,x1], [y0,y1], color=color, lw=1.6, alpha=0.9, zorder=3)

    lons_r = [float(CITIES[i][2]) for i in route]
    lats_r = [float(CITIES[i][1]) for i in route]
    gscatter(ax, lons_r, lats_r, color='steelblue', s=35, zorder=4)
    gscatter(ax, [lons_r[0]], [lats_r[0]], color='green', s=100, zorder=5, marker='*')

    pct = (hc_dist-dist)/hc_dist*100
    ax.set_title(f'Gen {gen}  |  {dist:.0f} km  ({pct:+.1f}% vs HC)',
                 fontsize=9, fontweight='bold',
                 color='darkgreen' if dist < hc_dist else 'gray')

sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn_r,
                            norm=plt.Normalize(0, max(DIST.flatten())))
sm.set_array([])
cbar = fig.colorbar(sm, ax=fig.axes, location='right', shrink=0.5, pad=0.02)
cbar.set_label('Leg distance (km) — green=short, red=long', fontsize=10)

fig.suptitle('Memetic GA Route Evolution — Snapshot per Generation\n'
             '(edges colored by leg distance | ★ = base | landmasses shown)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('fig6_route_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

Figure 6 saved.


In [13]:
# ============================================================
# CELL 11 — FIGURE 7: HC vs MEMETIC GA FINAL COMPARISON
# ============================================================
fig   = plt.figure(figsize=(18, 8))
gs_c  = GridSpec(1, 2, figure=fig)

def draw_final(fig, spec, route, dist, title, color):
    ax = make_ph_ax(fig, spec)
    for k in range(N):
        x0=float(CITIES[route[k]][2]);       y0=float(CITIES[route[k]][1])
        x1=float(CITIES[route[(k+1)%N]][2]); y1=float(CITIES[route[(k+1)%N]][1])
        ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.8))
    lons_r=[float(CITIES[i][2]) for i in route]
    lats_r=[float(CITIES[i][1]) for i in route]
    gscatter(ax, lons_r, lats_r, color=color, s=55, zorder=5, alpha=0.9)
    gscatter(ax, [lons_r[0]], [lats_r[0]], color='black', s=220, zorder=6, marker='*')
    for i,idx in enumerate(route):
        gannotate(ax, f'{i+1}.{CITIES[idx][0]}',
                  float(CITIES[idx][2]), float(CITIES[idx][1]),
                  fontsize=6.5, xytext=(4,2), textcoords='offset points')
    ax.set_title(title, fontsize=12, fontweight='bold')
    return ax

draw_final(fig, gs_c[0], hc_route, hc_dist,
           f'Hill Climbing — {hc_dist:.0f} km\n(10 restarts, local optimum)',
           'steelblue')
ax_r = draw_final(fig, gs_c[1], ga_route, ga_dist,
                  f'Memetic GA — {ga_dist:.0f} km\n({(hc_dist-ga_dist)/hc_dist*100:.1f}% shorter than HC  ✅)',
                  'crimson')
ax_r.text(0.02, 0.04,
          f'GA saved {hc_dist-ga_dist:.0f} km over HC',
          transform=ax_r.transAxes, fontsize=11,
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle('HC vs Memetic GA — Final Route Comparison\n'
             '(arrows = flight direction | numbers = visit order | ★ = base | landmasses shown)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_route_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

Figure 7 saved.


In [14]:
# ============================================================
# CELL 12 — FIGURE 8: PARAMETER SENSITIVITY
# ============================================================
print('Running sensitivity sweep (~1-2 min)...')
mutation_rates=[0.005,0.01,0.02,0.05,0.10]
pop_sizes=[20,40,80,120,160]
FAST_GENS=50
mr_results, pop_results = [], []

for mr in mutation_rates:
    random.seed(0); np.random.seed(0)
    pop=[two_opt(r) for r in init_pop(80)]
    for _ in range(FAST_GENS):
        pop.sort(key=fitness,reverse=True)
        np2=pop[:ELITES]
        while len(np2)<80:
            p1=tournament_select(pop,TOURNEY_K); p2=tournament_select(pop,TOURNEY_K)
            ch=ordered_crossover(p1,p2) if random.random()<CR else p1[:]
            np2.append(two_opt(swap_mutate(ch,mr)))
        pop=np2
    pop.sort(key=fitness,reverse=True)
    mr_results.append(route_distance(pop[0]))
    print(f'  MR={mr:.3f} → {mr_results[-1]:.0f} km')

for ps in pop_sizes:
    random.seed(0); np.random.seed(0)
    pop=[two_opt(r) for r in init_pop(ps)]
    for _ in range(FAST_GENS):
        pop.sort(key=fitness,reverse=True)
        k=min(TOURNEY_K,ps)
        np2=pop[:min(ELITES,ps)]
        while len(np2)<ps:
            p1=tournament_select(pop,k); p2=tournament_select(pop,k)
            ch=ordered_crossover(p1,p2) if random.random()<CR else p1[:]
            np2.append(two_opt(swap_mutate(ch,MR)))
        pop=np2
    pop.sort(key=fitness,reverse=True)
    pop_results.append(route_distance(pop[0]))
    print(f'  POP={ps} → {pop_results[-1]:.0f} km')

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot([str(m) for m in mutation_rates],mr_results,'o-',color='darkgreen',lw=2,ms=9,markerfacecolor='lime')
for x,y in enumerate(mr_results): axes[0].text(x,y-20,f'{y:.0f}km',ha='center',fontsize=9)
axes[0].axhline(hc_dist,color='orange',ls='--',lw=1.5,label=f'HC best ({hc_dist:.0f} km)')
axes[0].set_xlabel('Mutation Rate'); axes[0].set_ylabel('Best Route Distance (km)')
axes[0].set_title('Sensitivity: Mutation Rate\n(Memetic GA, pop=80, 50 gens)',fontsize=11,fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True,alpha=0.3)

axes[1].plot(pop_sizes,pop_results,'s-',color='darkblue',lw=2,ms=9,markerfacecolor='skyblue')
for x,y in zip(pop_sizes,pop_results): axes[1].text(x,y-20,f'{y:.0f}km',ha='center',fontsize=9)
axes[1].axhline(hc_dist,color='orange',ls='--',lw=1.5,label=f'HC best ({hc_dist:.0f} km)')
axes[1].set_xlabel('Population Size'); axes[1].set_ylabel('Best Route Distance (km)')
axes[1].set_title('Sensitivity: Population Size\n(Memetic GA, MR=0.02, 50 gens)',fontsize=11,fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('fig8_sensitivity.png',dpi=150,bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

Running sensitivity sweep (~1-2 min)...
  MR=0.005 → 6011 km
  MR=0.010 → 6011 km
  MR=0.020 → 6011 km
  MR=0.050 → 6011 km
  MR=0.100 → 6011 km
  POP=20 → 6011 km
  POP=40 → 6011 km
  POP=80 → 6011 km
  POP=120 → 6011 km
  POP=160 → 6011 km
Figure 8 saved.


In [16]:
# ============================================================
# CELL 13 — FIGURE 9: FULL SUMMARY DASHBOARD
# ============================================================
fig    = plt.figure(figsize=(22,14))
gs_d   = GridSpec(3,4,figure=fig,hspace=0.5,wspace=0.35)

# Convergence
ax_cv=fig.add_subplot(gs_d[0,:2])
ax_cv.plot(gens_range,best_hist,color='green',lw=2,label='GA Best')
ax_cv.plot(gens_range,avg_hist,color='steelblue',lw=1.2,ls='--',label='GA Avg')
ax_cv.axhline(hc_dist,color='orange',lw=1.8,ls='-.',label=f'HC Best ({hc_dist:.0f} km)')
ax_cv.axhline(ga_dist,color='darkgreen',lw=1.5,ls=':',label=f'GA Final ({ga_dist:.0f} km)')
ax_cv.fill_between(gens_range,best_hist,avg_hist,alpha=0.08,color='green')
ax_cv.set_title('Memetic GA Convergence',fontsize=11,fontweight='bold')
ax_cv.set_xlabel('Generation'); ax_cv.set_ylabel('km')
ax_cv.legend(fontsize=8); ax_cv.grid(True,alpha=0.3)

# Diversity
ax_dv=fig.add_subplot(gs_d[0,2])
ax_dv.plot(gens_range,div_hist,color='purple',lw=1.5)
ax_dv.fill_between(gens_range,div_hist,alpha=0.15,color='purple')
ax_dv.set_title('Population Diversity',fontsize=11,fontweight='bold')
ax_dv.set_xlabel('Generation'); ax_dv.set_ylabel('Avg diff'); ax_dv.grid(True,alpha=0.3)

# Stats box
ax_st=fig.add_subplot(gs_d[0,3]); ax_st.axis('off')
ax_st.text(0.05,0.95,
    f'RESULTS SUMMARY\n'
    f'━━━━━━━━━━━━━━━━━━━━\n'
    f'Dataset  : 51,043 waypoints\n'
    f'Countries: 209\n'
    f'PH wp    : {len(ph_df)}\n'
    f'TSP N    : {N}\n'
    f'Routes   : {math.factorial(N):.2e}\n'
    f'━━━━━━━━━━━━━━━━━━━━\n'
    f'HC best  : {hc_dist:.0f} km\n'
    f'GA best  : {ga_dist:.0f} km\n'
    f'Saved    : {hc_dist-ga_dist:.0f} km ({(hc_dist-ga_dist)/hc_dist*100:.1f}%)\n'
    f'━━━━━━━━━━━━━━━━━━━━\n'
    f'pop={POP_SIZE}, gens={GENS}\n'
    f'CR={CR}, MR={MR}\n'
    f'elites={ELITES}\n'
    f'local: 2-opt per offspring',
    transform=ax_st.transAxes,fontsize=8.5,va='top',fontfamily='monospace',
    bbox=dict(boxstyle='round',facecolor='#f0f8ff',alpha=0.9))

# HC route map
ax_hm=make_ph_ax(fig,gs_d[1,:2])
h_lons=[float(CITIES[i][2]) for i in hc_route+[hc_route[0]]]
h_lats=[float(CITIES[i][1]) for i in hc_route+[hc_route[0]]]
gplot(ax_hm,h_lons,h_lats,color='steelblue',lw=1.5,marker='o',ms=4,alpha=0.8,zorder=3)
gscatter(ax_hm,[h_lons[0]],[h_lats[0]],color='green',s=100,zorder=5,marker='*')
ax_hm.set_title(f'HC Route: {hc_dist:.0f} km',fontsize=11,fontweight='bold',color='steelblue')

# GA route map
ax_gm=make_ph_ax(fig,gs_d[1,2:])
g_lons=[float(CITIES[i][2]) for i in ga_route+[ga_route[0]]]
g_lats=[float(CITIES[i][1]) for i in ga_route+[ga_route[0]]]
gplot(ax_gm,g_lons,g_lats,color='crimson',lw=1.5,marker='o',ms=4,alpha=0.8,zorder=3)
gscatter(ax_gm,[g_lons[0]],[g_lats[0]],color='green',s=100,zorder=5,marker='*')
ax_gm.set_title(f'Memetic GA: {ga_dist:.0f} km ({(hc_dist-ga_dist)/hc_dist*100:.1f}% better)',
                fontsize=11,fontweight='bold',color='crimson')

# Mutation sensitivity
ax_mr=fig.add_subplot(gs_d[2,:2])
ax_mr.plot([str(m) for m in mutation_rates],mr_results,'o-',color='darkgreen',lw=2,ms=8)
ax_mr.axhline(hc_dist,color='orange',ls='--',lw=1.5,label='HC best')
ax_mr.set_xlabel('Mutation Rate'); ax_mr.set_ylabel('Best dist (km)')
ax_mr.set_title('Sensitivity: Mutation Rate',fontsize=11,fontweight='bold')
ax_mr.legend(fontsize=9); ax_mr.grid(True,alpha=0.3)

# Population sensitivity
ax_ps=fig.add_subplot(gs_d[2,2:])
ax_ps.plot(pop_sizes,pop_results,'s-',color='darkblue',lw=2,ms=8)
ax_ps.axhline(hc_dist,color='orange',ls='--',lw=1.5,label='HC best')
ax_ps.set_xlabel('Population Size'); ax_ps.set_ylabel('Best dist (km)')
ax_ps.set_title('Sensitivity: Population Size',fontsize=11,fontweight='bold')
ax_ps.legend(fontsize=9); ax_ps.grid(True,alpha=0.3)

fig.suptitle(
    'Memetic GA Philippine Aviation Route Optimization — Full Summary Dashboard\n'
    'Dataset: global_aviation_waypoints.csv  |  Kuznetsov Ch. 9–10  |  Landmasses: Cartopy / hull fallback',
    fontsize=14,fontweight='bold')
plt.savefig('fig9_dashboard.png',dpi=150,bbox_inches='tight')
plt.show()
print('Figure 9 — Final dashboard saved.')
print(f'\nAll 9 figures generated. GA beat HC by {hc_dist-ga_dist:.0f} km ({(hc_dist-ga_dist)/hc_dist*100:.1f}%).')

Figure 9 — Final dashboard saved.

All 9 figures generated. GA beat HC by 2123 km (26.1%).


---

## Discussion: Why the Memetic Algorithm Wins

### Why Plain GA Lost to HC (v1)
HC exhaustively checks all N(N-1)/2 = 190 swap pairs per iteration on N=20 — very aggressive for a small instance. Plain GA wasted generations on routes that were obviously fixable.

### Why Memetic GA Wins (v2)
2-opt polishes every offspring so no obviously bad routes survive. GA's crossover now combines locally-optimal solutions and uses population diversity to escape toward better global structure.

> *"Memetic Algorithms combine genetic search with local optimization."* — Kuznetsov, Ch. 9, Sec. 9.4.4, p. 201

### HC vs SA vs Memetic GA

| | HC | SA | Memetic GA |
|---|---|---|---|
| TSP N=20 | Local optima | Works, single solution | **Wins with 2-opt** |
| TSP N=100+ | Hopeless | Slow | Even stronger |
| Smooth landscape | Best choice | Overkill | Overkill |
| Complex scheduling | OK for small | Good | Best |

### Cartopy in Colab
Upload both this notebook and `global_aviation_waypoints.csv` to Colab. On first run Cartopy will download Natural Earth shapefiles (~10 MB each, auto-cached). All 9 figures will render with real Philippine coastlines, rivers, and lakes.